# CGLMP Inequalities

Juan Manuel Segura Guatibonza

In [7]:
include("SDP_QI.jl")
using CairoMakie

\begin{align*}
    \hat{X}_d &= \sum_{j=0}^{d-1} | j \rangle \langle j \oplus 1| \\
    \hat{F}_d &= \frac{1}{\sqrt{d}} \sum_{j,k} \omega_d^{jk} |j \rangle \langle k | \\
    \omega_d &= e^{2\pi i /d}
\end{align*}

The CGLMP ineuality is the following:

\begin{align*}
    I_d = \sum_{k=0}^{d/2 - 1} \left( 1 - \frac{2k}{d-1} \right) &\left[ (P(A_1 = B_1 + k) + P(B_1=A_2 + k +1) + P(A_2 = B_2+k) + (B_2 = A_1+k)) \right. \\
    & \left. - (P(A_1=B_1-k-1) + P(B_1=A_2-k) + P(A_2=B_2-k-1) + P(B_2=A_1-k-1)) \right]
\end{align*}

where

\begin{equation*}
    P(A_a = B_b + k) = \sum_{j=0}^{d-1} P(A_a=j, B_b= j+k \operatorname{mod} d)
\end{equation*}

Los elementos de medición de $A_1$ y $B_1$ son:
\begin{equation*}
    \Pi_{j|1} = |j\rangle \langle j | \quad , \quad j=0,\dots,d-1
\end{equation*}

Los elementos de medición de $A_2$ y $B_2$ son:
\begin{equation*}
    \Pi_{j|2} = \hat{M}_d|j\rangle \langle j | \hat{M}_d^\dagger \quad , \quad j=0,\dots,d-1
\end{equation*}

donde

\begin{equation*}
    \hat{M}_d = \hat{F}_d^{(1-\alpha)} \hat{X}_d^{-\alpha/2} \quad , \quad \alpha \in \left[0, 1 \right]
\end{equation*}

In [8]:
Threads.nthreads()

4

In [ ]:
function Optimal_CGLMP(d::Int)
    Alphas = LinRange(0, 1, 50)
    n = length(Alphas)
    Id_array = zeros(Float64, n)
    Entanglement = zeros(Float64, n)
    M_Incomp = zeros(Float64, n)

    Threads.@threads for i in 1:n
        α = Alphas[i]
        M = tunning_M(d, α)

        Proj_A1 = [begin 
            A1j = zeros(ComplexF64, d, d)
            A1j[j, j] = 1.0
            A1j  
        end for j in 1:d]

        Proj_A2 = [begin 
            ket = zeros(ComplexF64, d)
            ket[j] = 1.0
            ψ = M * ket
            ψ = ψ / norm(ψ)
            ψ * ψ'
        end for j in 1:d]

        Proj_A = [Proj_A1, Proj_A2]
        Proj_B = [Proj_A1, Proj_A2]

        Id_max, ρ = CGLMP_opt(Proj_A, Proj_B, d)

        Id_array[i] = Id_max
        r = RRE_PPT(ρ, d, d)
        Entanglement[i] = r / (1 + r)
        r = RRMI(Proj_A)
        M_Incomp[i] = r / (1 + r)

        println("Terminated process for α= $(α)")
    end

    CairoMakie.activate!(type = "pdf")

    set_theme!(Theme(
        fontsize = 18,
        fonts = (; regular = "CMU Serif"),
        Axis = (
            xgridvisible = true,
            ygridvisible = true,
            topspinevisible = true,
            rightspinevisible = true,
            spinewidth = 1.2,
        )
    ))

    f = Figure(size = (800, 900))

    ax1 = Axis(f[1, 1],
        ylabel = "CGLMP",
        title = "Optimal violation of the CGLMP inequality (d=$(d))",
        xticks = 0:0.2:1
    )

    ax2 = Axis(f[2, 1],
        ylabel = "Entanglement",
        xticks = 0:0.2:1
    )

    ax3 = Axis(f[3, 1],
        xlabel = L"\alpha",
        ylabel = "Incompatibility",
        xticks = 0:0.2:1
    )

    linkxaxes!(ax1, ax2, ax3)

    lines!(ax1, Alphas, Id_array, linewidth = 2)
    scatter!(ax1, Alphas, Id_array, markersize = 6)

    lines!(ax2, Alphas, Entanglement, linewidth = 2)
    scatter!(ax2, Alphas, Entanglement, markersize = 6)

    lines!(ax3, Alphas, M_Incomp, linewidth = 2)
    scatter!(ax3, Alphas, M_Incomp, markersize = 6)

    hidexdecorations!(ax1, ticks=false, grid=false)
    hidexdecorations!(ax2, ticks=false, grid=false)

    xlims!(ax3, 0, 1)

    rowgap!(f.layout, 10)

    save("resultsCGLMP/cglmp_d$(d).pdf", f)

    return f
end

Optimal_CGLMP (generic function with 1 method)

In [11]:
Optimal_CGLMP(2)

Terminated process for α= 0.0
Terminated process for α= 0.7755102040816326
Terminated process for α= 0.5306122448979592
Terminated process for α= 0.2653061224489796
Terminated process for α= 0.2857142857142857
Terminated process for α= 0.02040816326530612
Terminated process for α= 0.7959183673469388
Terminated process for α= 0.5510204081632653
Terminated process for α= 0.04081632653061224
Terminated process for α= 0.30612244897959184
Terminated process for α= 0.8163265306122449
Terminated process for α= 0.5714285714285714
Terminated process for α= 0.32653061224489793
Terminated process for α= 0.8367346938775511
Terminated process for α= 0.061224489795918366
Terminated process for α= 0.5918367346938775
Terminated process for α= 0.3469387755102041
Terminated process for α= 0.8571428571428571
Terminated process for α= 0.3673469387755102
Terminated process for α= 0.08163265306122448
Terminated process for α= 0.10204081632653061
Terminated process for α= 0.8775510204081632
Terminated proces